In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_postgres import PGVector, PGEngine, PGVectorStore, Column
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool
from langchain.agents import create_agent
import os 

load_dotenv()   #.env 파일에 저장되어있는 api 키 가져오기

True

### 병렬 도구 호출 테스트용 간단한 도구 만들어보기


In [2]:
@tool
def get_weather(city:str) -> str:
    """ 도시의 현재 날씨를 조회"""
    weather_table = {"서울" : "맑음 28도", "부산": "흐림27도", "강원도" : "비 25도"}
    return f"{city}: {weather_table[city]}"

In [23]:
# 1. model 설정
model = init_chat_model("openai:gpt-6-luna", 
                        reasoning_effort="none",
                        model_kwargs={"parallel_tool_calls":True})

In [24]:
agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="너는 사용자 질문에 맞춰서 답변을 친절하게 해주는 챗봇이야."
)

# agent 호출용 함수
def ask(agent, question):
    response = agent.invoke({"messages" : [ HumanMessage(question) ]})
    return response

In [25]:
result = ask(agent, "서울과 부산 날씨 알려줘")
result['messages']

[HumanMessage(content='서울과 부산 날씨 알려줘', additional_kwargs={}, response_metadata={}, id='c983468a-3f77-47a5-b3f9-8af160066795'),
 AIMessage(content=[{'arguments': '{"city":"서울"}', 'call_id': 'call_6PbYgk5sjvJiMHdBQYW160s1', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0cc9342899128ce1006ab9f7b0f11087d097dae5afe480b694', 'status': 'completed'}, {'arguments': '{"city":"부산"}', 'call_id': 'call_nJoD7eo06zDPq8MC0P21D9jK', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0cc9342899128ce1006ab9f7b0f12487d0893e58f82e56a581', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0cc9342899128ce1006ab9f7afead887d08319c4314df50ae7', 'created_at': 1790572463.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0cc9342899128ce1006ab9f7afead887d08319c4314df50ae7', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'c

In [26]:
# model 을 병렬 도구 호출로 바꾸기
# 1. 모델 설정
model = init_chat_model("openai:gpt-6-luna", 
                        reasoning_effort="none", 
                        model_kwargs={"parallel_tool_calls": True}
                        )

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="너는 사용자 질문에 맞춰서 답변을 친절하게 해주는 챗봇이야."
)

result = ask(agent, "서울과 부산, 강원도 날씨 알려줘")
result['messages']

[HumanMessage(content='서울과 부산, 강원도 날씨 알려줘', additional_kwargs={}, response_metadata={}, id='f9035072-5a83-4ce4-af90-8a188dd78fd7'),
 AIMessage(content=[{'arguments': '{"city":"서울"}', 'call_id': 'call_rL3msclEm11XMNNM4qYf7k5V', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0c03ad764405149f006ab9f7b4045887d0a7ce2b06fba4ca24', 'status': 'completed'}, {'arguments': '{"city":"부산"}', 'call_id': 'call_oh7Np5rRqXYR8w0sxKeFdYEw', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0c03ad764405149f006ab9f7b4046887d0ac3188de3462d02b', 'status': 'completed'}, {'arguments': '{"city":"강원도"}', 'call_id': 'call_6o5GeYtISsCL1k1TYsSSHUwe', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0c03ad764405149f006ab9f7b4047087d09a6a4de84b5b24e1', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0c03ad764405149f006ab9f7b3068c87d08937deb25063068e', 'created_at': 1790572467.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'defa

In [27]:
# 병렬과 순차가 섞여 있는 경우 

@tool                                                                               
def mean_temperature(                                                               
    seoul: int,                                                                     
    busan: int,                                                                     
    gangwon: int,                                                                   
) -> str:                                                                           
    """서울, 부산, 강원도의 기온을 입력받아 평균 기온을 계산한다."""                
    temperatures = [seoul, busan, gangwon]                                          
    average = sum(temperatures) / len(temperatures)                                 
                                                                                    
    return f"세 지역의 평균 기온은 {average:.1f}도입니다."

In [28]:
# model 을 병렬 도구 호출로 바꾸기
# 1. 모델 설정
model = init_chat_model("openai:gpt-6-luna", 
                        reasoning_effort="none", 
                        model_kwargs={"parallel_tool_calls": True}
                        )

agent = create_agent(
    model=model,
    tools=[get_weather,mean_temperature],
    system_prompt="너는 사용자 질문에 맞춰서 답변을 친절하게 해주는 챗봇이야."
)

result = ask(agent, "서울과 부산, 강원도 날씨 알려줘, 그리고 세 지역의 평균 기온을 알려줘")
result['messages']

[HumanMessage(content='서울과 부산, 강원도 날씨 알려줘, 그리고 세 지역의 평균 기온을 알려줘', additional_kwargs={}, response_metadata={}, id='011cc707-e192-491b-aa91-068c76cb469e'),
 AIMessage(content=[{'arguments': '{"city":"서울"}', 'call_id': 'call_EmWMG9M8w13TU0JiTzmaDIaZ', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0887f1d28834e617006ab9f7b7b1cc87d084a75b228d8fb0c0', 'status': 'completed'}, {'arguments': '{"city":"부산"}', 'call_id': 'call_uouu54yUpdy3cC2PFdBkigOt', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0887f1d28834e617006ab9f7b7b1dc87d0b7d93dbc383c5a92', 'status': 'completed'}, {'arguments': '{"city":"강원도"}', 'call_id': 'call_X8qcq1iWXTjqXEchDIpTrLzF', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0887f1d28834e617006ab9f7b7b1e487d0a999d121b6c930c8', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0887f1d28834e617006ab9f7b6d1c887d08678f4e2cc41fe8e', 'created_at': 1790572470.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response',

In [29]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from collections.abc import Callable

In [30]:
# 에러처리용 도구 추가
@wrap_tool_call
def handle_tool_error(request, handler):
    try :
        return handler(request)
    except : 
        return ToolMessage(

            content="도시의 날씨 정보가 없습니다",
            tool_call_id = request.tool_call["id"],
            name=request.tool_call["name"],
            status='error'
        )


In [31]:
agent = create_agent(
    model=model,
    tools=[get_weather,mean_temperature],
    system_prompt="너는 사용자 질문에 맞춰서 답변을 친절하게 해주는 챗봇이야.",
    middleware=[handle_tool_error]
)

result = ask(agent, "서울과 부산, 강원도 날씨 알려줘, 그리고 세 지역의 평균 기온을 알려줘")
result['messages']

[HumanMessage(content='서울과 부산, 강원도 날씨 알려줘, 그리고 세 지역의 평균 기온을 알려줘', additional_kwargs={}, response_metadata={}, id='d92b1379-6077-48eb-8db5-9558dceaeb14'),
 AIMessage(content=[{'arguments': '{"city":"서울"}', 'call_id': 'call_XWobSBXG2rLkK8f4jSZO3f5j', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0cd89591add58bfe006ab9f7bccaa087d0a2678811cd2883f0', 'status': 'completed'}, {'arguments': '{"city":"부산"}', 'call_id': 'call_oHHB82RUSOezGr5hTrcvFq39', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0cd89591add58bfe006ab9f7bccab487d0af86a10e7d51619f', 'status': 'completed'}, {'arguments': '{"city":"강원도"}', 'call_id': 'call_h3aevaQvdmzCiTS4aMIEqhjz', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0cd89591add58bfe006ab9f7bccac087d0a8a4348eb3620391', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0cd89591add58bfe006ab9f7bbfe4c87d0b106386306e56e5d', 'created_at': 1790572476.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response',

### 무한 반복 막기
- Recursion_limit
- 에이전트가 도구를 무한 반복 호출하지 않도록 최대 단계수를 제한

In [32]:
from langchain.agents.middleware import ModelCallLimitMiddleware


agent = create_agent(

    model=model,
    system_prompt="너는 사용자 질문에 맞춰서 답변을 친절하게 해주는 챗봇이야.",
    middleware=[handle_tool_error, ModelCallLimitMiddleware(run_limit=5, exit_behavior='end')]
)

result = ask(agent, "서울과 부산, 강원도 날씨 알려줘, 그리고 세 지역의 평균 기온을 알려줘. 최대 10까지 호출해봐")
result['messages']

OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': "Invalid value for 'parallel_tool_calls': 'parallel_tool_calls' is only allowed when 'tools' are specified.", 'type': 'invalid_request_error', 'param': 'parallel_tool_calls', 'code': None}}

In [15]:
# 무한 반복 되도록 날씨 도구를 망가뜨려보기

@tool
def get_weather(city : str) -> str:
    """도시의 현재 날씨를 조회한다. """
    weather_table = {"서울" : "오늘은 기분이 좋아요", "부산" : "점심 메뉴는 돈까스입니다", "강원도": "강원도 메뉴는 닭갈비"}
    # return f"{city}: {weather_table.get(city, "맑음 24도")}"
    return f"{city}: {weather_table[city]}"

agent = create_agent(

    model=model,
    system_prompt="너는 사용자 질문에 맞춰서 답변을 친절하게 해주는 챗봇이야. 답변이 이상할 경우 도구 호출을 반복해봐",
    middleware=[handle_tool_error, ModelCallLimitMiddleware(run_limit=5, exit_behavior='end')]
)

result = ask(agent, "서울과 부산, 강원도 날씨 알려줘, 그리고 세 지역의 평균 기온을 알려줘.")
result['messages']

OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': "Invalid value for 'parallel_tool_calls': 'parallel_tool_calls' is only allowed when 'tools' are specified.", 'type': 'invalid_request_error', 'param': 'parallel_tool_calls', 'code': None}}

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def send_mail(to: str, subject: str, body: str) -> str:
    """이메일 전송 (실습용: 실제 메일을 보내지 않고 결과 문자열만 반환)"""
    return f"{to}에게 이메일 전송 완료"


In [ ]:
# 승인 대기와 재개에 사용할 체크포인터를 처음부터 연결합니다.
memory = InMemorySaver()
hitl_agent = create_agent(
    model=model,
    tools=[send_mail],
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={"send_mail": {"allowed_decisions": ["approve", "reject"]}}
    )],
    checkpointer=memory,
)


In [ ]:
def ask(agent, question, config):
    return agent.invoke({"messages": [HumanMessage(question)]}, config=config)


In [ ]:
# 실제 승인 대기는 아래 Cell 21에서 한 번만 시작합니다.
print("체크포인터 연결 완료. Cell 20의 config 설정 후 Cell 21을 실행하세요.")


In [ ]:
# 중요: 여기서 에이전트를 새로 만들지 않습니다. 위의 hitl_agent와 memory를 그대로 이어 씁니다.


In [ ]:
# 동일한 thread_id를 사용해야 대기 중이던 interrupt 상태를 불러올 수 있습니다.
config = {"configurable": {"thread_id": "sk-net-encore-email-1"}}


In [ ]:
# 승인 대기 상태 생성: 먼저 Cell 18을 실행한 뒤 이 셀을 실행합니다.
result = ask(hitl_agent, "yttes133@gmail.com에게 제목은 테스트, 본문은 미들웨어 테스트중", config=config)
result


In [ ]:
# 사람이 승인: Cell 21에서 interrupt가 발생한 다음, 같은 체크포인터와 thread_id로 재개합니다.
after_approval = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
)
after_approval["messages"]


### 사용할 수 잇는 미들웨어
1. 도구를 사용 하기전에 사람이 승인(도구 사용전 승인)
2. 대화가 길어지면 자동요약(대화 최적화)

In [39]:
# 도시별 날씨 조회 도구
@tool
def get_weather(city : str) -> str:
    """도시의 현재 날씨를 조회한다. """
    weather_table = {"서울" : "맑음 28도", "부산" : "흐림 27도", "강원도": "비 25도"}
    return f"{city}: {weather_table.get(city, "맑음 24도")}"
    # return f"{city}: {weather_table[city]}"   # 일부러 에러 띄우는 용도였다


# 병렬과 순차가 섞여 있는 경우 
@tool                                                                               
def mean_temperature(
    temperatures: list[float]                                                                
) -> str:                                                                           
    """도시들의 기온을 입력받아 평균 기온을 계산한다."""                
    average = sum(temperatures) / len(temperatures)                                 
                                                                                    
    return f"세 지역의 평균 기온은 {average:.1f}도입니다."

from langchain.agents.middleware import SummarizationMiddleware

# 대화가 길어졌을 때 사용할 자동 요약 미들웨어
summarization = SummarizationMiddleware(
    model=model,
    trigger=("messages", 10),  # 메시지 10개 이상이면 요약
    keep=("messages", 4),       # 최근 메시지 4개는 유지
)

agent = create_agent(
    model=model,
    tools=[get_weather, mean_temperature],
    system_prompt="""
너는 사용자 질문에 맞춰서 답변을 친절하게 해주는 챗봇이야.
이전 대화의 문맥을 참고해서 답변해.
""",
    middleware=[
        summarization,
    ],
)

result = ask(
    agent,
    "서울과 부산, 대전 날씨를 알려줘. 이전에 이야기한 내용도 반영해서 평균 기온을 알려줘.",
    config
)

result["messages"]

[HumanMessage(content='서울과 부산, 대전 날씨를 알려줘. 이전에 이야기한 내용도 반영해서 평균 기온을 알려줘.', additional_kwargs={}, response_metadata={}, id='bdaad569-b13b-47ec-a4a4-1cf48e9ebbc6'),
 AIMessage(content=[{'arguments': '{"city":"서울"}', 'call_id': 'call_mFGFkHoeHmSMNCZpNfgKvetv', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0b873cff998d6d34006ab9fd5a869c87d0ba0dc1fd3a54ffd4', 'status': 'completed'}, {'arguments': '{"city":"부산"}', 'call_id': 'call_LrJhbVRHG5OfiIumX60VB9OP', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0b873cff998d6d34006ab9fd5a86b087d0b70d54b58cf91d35', 'status': 'completed'}, {'arguments': '{"city":"대전"}', 'call_id': 'call_nPkf5dGlvjB2NLJLGqONe7w6', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0b873cff998d6d34006ab9fd5a86bc87d0a6ead74f00cdda0a', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0b873cff998d6d34006ab9fd59af7487d0a666640ae9736143', 'created_at': 1790573913.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 're